# GTEx model building with PCA, NFM and ICA

💡 **Environment:** `clamp-analyses`  

## Libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pyprojroot.here import here

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, NMF, FastICA
import joblib

## Input

In [2]:
gtex_data = pd.read_csv(here("output/gtex/df_gtex_fbm_filt.csv"), index_col=0).astype(np.float32)

In [8]:
K = pd.read_csv(here("output/gtex/CLAMP_K_gtex.csv"))
n_components = int(K["CLAMP_K_gtex"].iloc[0])

print(f"Number of components: {n_components}")

Number of components: 578


## Output

In [9]:
pca_dir = Path(here("output/gtex/pca"))
ica_dir = Path(here("output/gtex/ica"))
nmf_dir = Path(here("output/gtex/nmf"))
for d in [pca_dir, ica_dir, nmf_dir]:
    d.mkdir(parents=True, exist_ok=True)

# PCA

In [10]:
X = gtex_data.T  # samples x genes

pca = PCA(n_components=n_components, svd_solver="auto", random_state=123)
W = pca.fit_transform(X)   # samples x comps
H = pca.components_               # comps x genes

pc_names = [f"PC{i+1}" for i in range(W.shape[1])]

gtex_pca_scores   = pd.DataFrame(W, index=gtex_data.columns, columns=pc_names)   # samples x comps
gtex_pca_loadings = pd.DataFrame(H, index=pc_names, columns=gtex_data.index)    # comps x genes
gtex_pca_B = gtex_pca_scores.T
gtex_pca_B.index.name = "PC"

gtex_pca_B.to_pickle(pca_dir / "gtex_pca_B.pkl")
gtex_pca_scores.to_pickle(pca_dir / "gtex_pca_scores.pkl")
gtex_pca_loadings.to_pickle(pca_dir / "gtex_pca_loadings.pkl")

In [11]:
gtex_pca_B.head()

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,...,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
PC,,,,,,,,,,,,,,,,,,,,,
PC1,84.419952,-71.704193,61.518253,93.491264,-70.403038,-10.793831,125.473923,73.683098,39.082565,31.023188,...,-49.669159,20.674223,73.425797,-23.001375,16.341793,89.849876,-15.991742,31.217775,-84.271935,28.372345
PC2,-34.006119,-18.352537,-31.801624,-11.326031,-17.874212,-47.900562,1.370198,-31.237597,-30.852791,-43.721626,...,-6.907615,-5.560450,1.115206,-17.848867,-6.578469,8.008780,-43.369492,-15.860476,-13.706820,-35.314190
PC3,-4.317819,-55.056732,-29.385828,-20.059416,-28.841183,-4.676594,-2.078792,34.484035,-6.360053,12.579873,...,-46.490322,-7.465433,22.277159,-1.794269,-50.160332,-9.280685,24.121758,-61.771927,-71.058647,-31.196804
PC4,10.149578,-31.034983,21.364086,39.329445,19.357540,28.099674,32.057682,-59.406021,9.050756,-47.222691,...,3.812490,-6.047695,11.740247,-10.784235,8.594497,5.633564,-82.291985,21.471979,-40.803600,18.156662
PC5,8.714474,-29.465523,3.403795,9.115075,13.808648,11.056884,15.000153,29.756916,13.495225,36.101669,...,-9.150455,-19.181095,23.800186,14.506798,-11.331547,3.911417,12.602872,-10.915432,-50.054386,-5.755920


# ICA

In [12]:
X = gtex_data.T  # samples x genes

ica = FastICA(n_components=n_components, random_state=123, max_iter=2000)
W = ica.fit_transform(X)  # samples x comps
H = ica.mixing_.T                # comps x genes

ic_names = [f"IC{i+1}" for i in range(W.shape[1])]

gtex_ica_scores   = pd.DataFrame(W, index=gtex_data.columns, columns=ic_names)   # samples x comps
gtex_ica_loadings = pd.DataFrame(H, index=ic_names, columns=gtex_data.index)    # comps x genes
gtex_ica_B = gtex_ica_scores.T
gtex_ica_B.index.name = "IC"

gtex_ica_B.to_pickle(ica_dir / "gtex_ica_B.pkl")
gtex_ica_scores.to_pickle(ica_dir / "gtex_ica_scores.pkl")
gtex_ica_loadings.to_pickle(ica_dir / "gtex_ica_loadings.pkl")

In [13]:
gtex_ica_B.head()

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,...,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
IC,,,,,,,,,,,,,,,,,,,,,
IC1,1.818188,-2.894583,-2.090720,-0.438408,0.100919,-0.856781,0.674642,4.930187,-0.389865,0.711956,...,0.837943,0.391684,0.951789,0.448467,0.214657,-0.046541,4.835483,-0.048097,0.386790,0.852790
IC2,0.598212,-0.689172,0.622834,1.796673,0.440248,0.312387,2.327956,1.355525,0.172466,0.112952,...,0.030650,0.212311,1.281829,0.140839,-0.399821,-0.300892,-0.362571,0.327991,-0.674140,0.794769
IC3,-0.518521,0.486116,0.348091,0.189927,-0.084445,-0.217268,0.156711,0.121109,0.035067,0.018844,...,-0.176499,-0.260183,0.870851,-0.248130,0.091180,-0.045680,0.268200,-0.050999,-0.487501,0.201026
IC4,0.594633,0.553572,-0.203302,0.047741,1.039842,0.142532,-1.170967,0.247792,0.459793,0.444938,...,0.298489,-0.142048,-0.171276,-0.325074,1.173493,0.280409,1.378153,0.676624,0.392438,-0.088661
IC5,0.671957,0.398128,0.615031,0.405379,0.403113,0.536337,0.341667,0.817012,0.311953,0.690237,...,-0.098120,-0.245105,-0.411067,0.065750,-0.044758,-0.063053,-0.125928,-0.119144,0.218688,0.220810


# NMF

In [14]:
# non negative input need it
gene_min = gtex_data.min(axis=1)                 
gtex_data_nmf = gtex_data.sub(gene_min, axis=0) 

In [15]:
X = gtex_data_nmf.T  # samples x genes (non-negative)

nmf = NMF(n_components=n_components, init="nndsvd", random_state=123, max_iter=1000)
W = nmf.fit_transform(X)  # samples x comps
H = nmf.components_       # comps x genes

lv_names = [f"LV{i+1}" for i in range(W.shape[1])]

gtex_nmf_scores   = pd.DataFrame(W, index=gtex_data.columns, columns=lv_names)  # samples x comps
gtex_nmf_loadings = pd.DataFrame(H, index=lv_names, columns=gtex_data.index)   # comps x genes
gtex_nmf_B = gtex_nmf_scores.T
gtex_nmf_B.index.name = "LV"

gtex_nmf_B.to_pickle(nmf_dir / "gtex_nmf_B.pkl")
gtex_nmf_scores.to_pickle(nmf_dir / "gtex_nmf_scores.pkl")
gtex_nmf_loadings.to_pickle(nmf_dir / "gtex_nmf_loadings.pkl")

/home/msubirana/miniconda3/envs/clamp-analyses/lib/python3.11/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


In [16]:
gtex_nmf_B.head()

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,...,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
LV,,,,,,,,,,,,,,,,,,,,,
LV1,0.457428,0.0,0.455101,0.532474,0.248576,0.325243,0.620044,0.387330,0.455219,0.413403,...,0.305180,0.509628,0.453751,0.409129,0.685056,0.518297,0.370409,0.621415,0.027878,0.460471
LV2,0.000000,0.0,0.000000,0.004031,0.010839,0.010275,0.000000,0.001008,0.000000,0.000000,...,0.007252,0.005584,0.006543,0.004496,0.011268,0.000000,0.000000,0.001144,0.022147,0.000000
LV3,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.003729,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LV4,0.026562,0.0,0.018743,0.000000,0.000000,0.000000,0.000000,0.000000,0.014544,0.006821,...,0.002121,0.023762,0.042345,0.004337,0.000000,0.000000,0.016795,0.023736,0.000000,0.000000
LV5,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.091857,0.006260,0.489079,...,0.000000,0.000752,0.000000,0.005178,0.000000,0.000000,0.176650,0.000000,0.000000,0.000000
